# RSA finalists — full quality + real native CPU benchmark

This is the decisive follow-up experiment. It runs the full 44k / 3-seed semantic-quality benchmark, then exports **real held-out item codes and real learned predicate programs** for the four finalists and benchmarks them in Rust.

Finalists: **FP32 linear, PQ64, BBQ1-LS2-int4, RSA2**.

The Rust benchmark tiles held-out test rows only to enlarge the resident working set; it does not generate synthetic item values or synthetic predicates.


In [ ]:
#@title 1) Settings
FULL_QUALITY = True #@param {type:"boolean"}
RESIDENT_ITEMS = 500000 #@param {type:"integer"}
RUST_REPEATS = 5 #@param {type:"integer"}
print('FULL_QUALITY =', FULL_QUALITY)
print('RESIDENT_ITEMS =', RESIDENT_ITEMS)


In [ ]:
#@title 2) Clone repo and install dependencies
import os, pathlib, shutil, subprocess
ROOT=pathlib.Path('/content/ras')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run(['pip','install','-q','-e','.','faiss-cpu'],check=True)
print('repo:', subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip())


In [ ]:
#@title 3) Run semantic-quality experiment
import os, subprocess, time
os.chdir('/content/ras')
cfg='configs/binary_bbq.yaml' if FULL_QUALITY else 'configs/binary_bbq_smoke.yaml'
print('config:',cfg)
t0=time.time()
subprocess.run(['python','-m','experiments.binary_bbq_predicates','--config',cfg],check=True)
print(f'quality run finished in {(time.time()-t0)/60:.1f} min')


In [ ]:
#@title 4) Show quality Pareto table
from pathlib import Path
import pandas as pd, json
runs=sorted([p for p in Path('/content/ras/results').iterdir() if p.is_dir() and '_binary_' in p.name],key=lambda p:p.stat().st_mtime)
run=runs[-1]
print('quality run:',run)
pareto=pd.read_csv(run/'pareto_at_20pct.csv')
focus=['linear_fp32','pq64_linear_lut','bbq1_ls2_int4q','rsa2_random']
display(pareto[pareto.method.isin(focus)][['method','bytes_per_item','program_bytes_per_concept','recall','purity']].sort_values('recall',ascending=False))
pred=pd.read_csv(run/'predicate_metrics.csv')
display(pred[pred.method.isin(focus)].groupby('method')[['f1','ap']].mean().sort_values('f1',ascending=False))


In [ ]:
#@title 5) Export real first-seed native assets
import os, subprocess, shutil, pathlib
os.chdir('/content/ras')
assets=pathlib.Path('/content/rsa_native_assets')
if assets.exists(): shutil.rmtree(assets)
subprocess.run(['python','-m','experiments.export_native_finalists','--config',cfg,'--out-dir',str(assets)],check=True)
print((assets/'manifest.json').read_text())


In [ ]:
#@title 6) Install Rust if needed and compile
import shutil, subprocess, os
if shutil.which('cargo') is None:
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','cargo','rustc'],check=True)
print(subprocess.check_output(['rustc','--version']).decode().strip())
os.chdir('/content/ras')
subprocess.run(['cargo','test','--release','--manifest-path','rust/semantic_engine/Cargo.toml'],check=True)


In [ ]:
#@title 7) Benchmark real learned finalists in Rust
import subprocess, time, os
os.chdir('/content/ras')
native_out='/content/rsa_actual_finalists_results.csv'
t0=time.time()
subprocess.run([
 'cargo','run','--release','--manifest-path','rust/semantic_engine/Cargo.toml','--bin','actual','--',
 '--assets','/content/rsa_native_assets',
 '--resident-items',str(RESIDENT_ITEMS),
 '--repeats',str(RUST_REPEATS),
 '--out',native_out
],check=True)
print(f'native benchmark finished in {(time.time()-t0)/60:.1f} min')


In [ ]:
#@title 8) Throughput results and speedups
import pandas as pd
sysdf=pd.read_csv('/content/rsa_actual_finalists_results.csv')
s=sysdf[sysdf.candidates==100000].copy()
tab=s.pivot(index='representation',columns='predicates',values='million_candidates_per_s')
display(tab)
for p in [1,2,4,8]:
    z=s[s.predicates==p].set_index('representation').million_candidates_per_s
    if 'fp32_linear' in z:
        print(f'P={p}: BBQ/FP32={z.get("bbq1_ls2_int4q",float("nan"))/z["fp32_linear"]:.2f}x  RSA2/FP32={z.get("rsa2_random",float("nan"))/z["fp32_linear"]:.2f}x  PQ/FP32={z.get("pq64_linear_lut",float("nan"))/z["fp32_linear"]:.2f}x')


In [ ]:
#@title 9) Quality × memory × real throughput table
q=pareto[pareto.method.isin(focus)][['method','bytes_per_item','program_bytes_per_concept','recall','purity']].copy()
speed=s[s.predicates==1][['representation','million_candidates_per_s']].rename(columns={'representation':'method'})
final=q.merge(speed,on='method',how='left')
display(final.sort_values('recall',ascending=False))
print('\nPredicate-store size at 10k concepts:')
final['predicate_store_10k_MB']=final.program_bytes_per_concept*10000/1e6
display(final[['method','predicate_store_10k_MB','bytes_per_item','recall','million_candidates_per_s']].sort_values('predicate_store_10k_MB'))


In [ ]:
#@title 10) Zip everything
import shutil, pathlib
bundle=pathlib.Path('/content/rsa_finalists_bundle')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
shutil.copytree(run,bundle/'quality')
shutil.copytree('/content/rsa_native_assets',bundle/'native_assets')
shutil.copy('/content/rsa_actual_finalists_results.csv',bundle/'actual_finalists_results.csv')
zip_path=shutil.make_archive('/content/rsa_finalists_full_native','zip',root_dir=str(bundle))
print(zip_path)
